# Euler–Lagrange with SageMath

Rewrite of `learn_sympy.ipynb`: derive EL via one-parameter variation, then specialize to shortest path.

Use conda env **`deploy_vllm`** (SageMath 10.x). In a Python kernel:
```python
from sage.all import *
```

In [9]:
from sage.all import *
from IPython.display import display, Math

## 1. Derive Euler–Lagrange via variation $\eta$

Sage tips:
- Build $F_\varepsilon = F\big(x,\, y+\varepsilon\eta,\, (y+\varepsilon\eta)'\big)$ **explicitly**. Only substituting $y\mapsto y_\varepsilon$ leaves the third slot as $y'$, so the $\eta'$ term vanishes.
- Partials appear as $D[1](F)$, $D[2](F)$ (cleaner than SymPy `Subs`).
- No built-in abstract IBP: write `ibp_on_eta` yourself; use `.coefficient(...)` (SymPy-style `Wild`/`match` is unreliable on derivatives).


In [ ]:
x = var("x")
y = function("y")
F = function("F")
eta = function("eta")

F_expr = F(x, y(x), diff(y(x), x))
# abstract F(..., y') cannot be integrated by Maxima — keep it formal
objective = integral(F_expr, x, hold=True)

eps = var("epsilon")
y_eps = y(x) + eps * eta(x)

# vary BOTH the y and y' slots
F_eps = F(x, y_eps, diff(y_eps, x))

# first-order Taylor in ε → F + ε (F_y η + F_y' η') + …
F_lin = F_eps.taylor(eps, 0, 1)
integrand = ((F_lin - F_expr) / eps).expand()
display(Math(latex(integrand)))

a, b = var("a b")


def ibp_on_eta(integrand, eta, x, a, b):
    """∫ U η' dx = [U η]_a^b − ∫ (dU/dx) η dx

    Sage has no built-in IBP for abstract integrands (and SR.wild().match
    often fails on derivatives). Use .coefficient(...) instead of SymPy Wild.
    `eta` should be the callable from function("eta"), not eta(x).
    """
    d_eta = diff(eta(x), x)
    # terms ~ η' → U ; terms ~ η → stay
    U = integrand.coefficient(d_eta)
    unchange = integrand.coefficient(eta(x)) * eta(x)

    boundary = (U * eta(x)).subs(x == b) - (U * eta(x)).subs(x == a)
    return (
        integral(unchange, x, a, b, hold=True)
        - integral(diff(U, x) * eta(x), x, a, b, hold=True)
        + boundary
    )


after = ibp_on_eta(integrand, eta, x, a, b)

# fixed endpoints η(a)=η(b)=0 → drop boundary
delta_J_fixed = after.subs({eta(a): 0, eta(b): 0})

# combine volume terms: ∫ (F_y − d/dx F_y') η dx
Fy = integrand.coefficient(eta(x))
U = integrand.coefficient(diff(eta(x), x))
E = Fy - diff(U, x)   # diff expands total derivative (chain rule)
combined = integral(E * eta(x), x, a, b, hold=True)
el_eq = E == 0

display(Math(latex(combined)))
display(Math(latex(el_eq)))
# Fundamental lemma: η arbitrary ⇒ E(x) = 0 (Euler–Lagrange)



<IPython.core.display.Math object>

## 2. Shortest path: $F = \sqrt{1+(y')^2}$

Substitute the concrete integrand into the abstract EL with `substitute_function`, then `desolve`.

In [ ]:
# arc-length density
dx = var("dx", domain="positive")
dy = diff(y(x), x) * dx
dL = sqrt(dx**2 + dy**2).simplify_full()  # type: ignore
dL_dx = (dL / dx).simplify_full()   # = sqrt(1 + y'(x)^2)

L = integral(dL_dx, x, a, b)
display(Math(latex(L)))

# F(x, y, y') → √(1+(y')²)  inside EL from above
E_arc = E.substitute_function(F, lambda *args: sqrt(1 + args[2]**2)).simplify_full()
el_arc = E_arc == 0
display(Math(latex(el_arc)))
# ⇒ y'' = 0 (straight line)

# analytic solution
sol = desolve(E_arc.numerator() == 0, y(x), ivar=x)   # _K1*x + _K2
display(Math(latex(sol)))

ya, yb = var("y_a y_b")
sol_bc = desolve(diff(y(x), x, 2) == 0, y(x), ivar=x, ics=[a, ya, b, yb])
display(Math(latex(sol_bc)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>